In [ ]:
import warnings
warnings.filterwarnings("ignore")

import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats as sp_stats
from scipy.stats import binomtest, mannwhitneyu, fisher_exact, kruskal
from IPython.display import display, HTML, Markdown

# ── Database connection ──
DB_PATH = "C:/Users/scgee/OneDrive/Documents/Projects/PatientPunk/pssd.db"
conn = sqlite3.connect(DB_PATH)

# ── Sentiment mapping ──
SENTIMENT_SCORE = {"positive": 1.0, "mixed": 0.5, "neutral": 0.0, "negative": -1.0}

def to_numeric(s):
    """Convert sentiment string to numeric score."""
    return SENTIMENT_SCORE.get(s, 0.0)

def classify_outcome(avg_score):
    """Classify user-level average into outcome category."""
    if avg_score > 0.7:
        return "positive"
    elif avg_score < -0.3:
        return "negative"
    return "mixed/neutral"

def wilson_ci(k, n, z=1.96):
    """Wilson score confidence interval for a proportion."""
    if n == 0:
        return 0.0, 0.0
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    margin = z * np.sqrt((p * (1 - p) + z**2 / (4 * n)) / n) / denom
    return max(0, center - margin), min(1, center + margin)

def nnt(treatment_rate, baseline_rate):
    """Number needed to treat. Returns None if rates are equal or inverted."""
    diff = treatment_rate - baseline_rate
    if diff <= 0:
        return None
    return round(1 / diff, 1)

# ── Chart defaults ──
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

# ── Filtering sets ──
GENERIC_TERMS = {
    "supplements", "medication", "treatment", "therapy", "drug", "drugs",
    "vitamin", "prescription", "pill", "pills", "dosage", "dose",
}

# Colors
COLORS = {"positive": "#2ecc71", "mixed/neutral": "#95a5a6", "negative": "#e74c3c"}


**Research Question:** "What treatments improve PSSD (Post-SSRI Sexual Dysfunction) once people have it?"

# PSSD Recovery Treatments: What Works According to the Community

**Abstract:** This analysis examines 585 treatment reports from 146 unique users in the r/PSSD subreddit to identify which interventions the community reports as helpful for recovering from PSSD (Post-SSRI Sexual Dysfunction). After excluding causative drugs (SSRIs, SNRIs, and related antidepressants whose negative sentiment reflects cause rather than treatment response), antihistamines and mast cell stabilizers emerge as the most promising recovery class, with ketotifen (88% positive), loratadine (57% positive), and cetirizine (67% positive) showing strong community endorsement. Dopamine-focused interventions (bupropion 38%, cabergoline 42%, pramipexole 50%) and lifestyle approaches (ketogenic diet 83%, psychedelic microdosing 78%) also show above-baseline positive rates. All findings should be interpreted cautiously given small sample sizes, self-selection, and the absence of controlled comparison groups.

## Data Exploration

The r/PSSD subreddit is a community for people experiencing persistent sexual dysfunction after discontinuing SSRIs and related medications. Most treatment discussion in this community is recovery-focused: users share what they have tried and whether it helped.

In [ ]:
# ── Data overview ──
total_reports = pd.read_sql("SELECT COUNT(*) as n FROM treatment_reports", conn).iloc[0,0]
total_users = pd.read_sql("SELECT COUNT(DISTINCT user_id) as n FROM treatment_reports", conn).iloc[0,0]
total_drugs = pd.read_sql("SELECT COUNT(DISTINCT drug_id) as n FROM treatment_reports", conn).iloc[0,0]

date_range = pd.read_sql(
    "SELECT date(MIN(p.post_date), 'unixepoch') as start_date, "
    "date(MAX(p.post_date), 'unixepoch') as end_date FROM posts p WHERE p.post_date IS NOT NULL", conn)

display(HTML(f'''
<div style="background: #f8f9fa; padding: 15px; border-radius: 8px; margin: 10px 0;">
<h3 style="margin-top:0">Dataset Overview</h3>
<table style="font-size: 14px;">
<tr><td><b>Data covers:</b></td><td>{date_range.iloc[0]["start_date"]} to {date_range.iloc[0]["end_date"]} (1 month)</td></tr>
<tr><td><b>Total treatment reports:</b></td><td>{total_reports:,}</td></tr>
<tr><td><b>Unique reporters:</b></td><td>{total_users}</td></tr>
<tr><td><b>Unique treatments mentioned:</b></td><td>{total_drugs}</td></tr>
</table>
</div>
'''))

### Filtering Methodology

This is a recovery-focused analysis. Because PSSD is *caused* by SSRIs and related antidepressants, these drugs dominate the treatment reports with overwhelmingly negative sentiment that reflects causation, not treatment response. Including them would contaminate the analysis.

**Causative drugs excluded:** SSRIs (sertraline, escitalopram/Lexapro, fluoxetine/Prozac, paroxetine, citalopram, vortioxetine/Brintellix), SNRIs (duloxetine, venlafaxine), and generic terms (antidepressant, SSRI, SNRI). Also excluded: finasteride and isotretinoin (other known PSSD-causing agents).

**Generic terms excluded:** supplements, medication, treatment, therapy, drug/drugs (categories, not actionable treatments).

**Duplicates merged:** dextromethorphan/DXM, cyproheptadine/ciproheptadine, quercetin/liposomal quercetin, cannabis/weed/marijuana, shrooms/psilocybin/microdosing/LSD/psychedelics (as "psychedelics" class).

**Non-treatment entries excluded:** alcohol, cocaine, coffee, seed, 75mg (ambiguous dosage reference), antibiotic (non-specific).

In [ ]:
# ── Build recovery treatment dataset with merges and exclusions ──
import math

CAUSAL_DRUGS = {
    'ssri', 'sertraline', 'lexapro', 'fluoxetine', 'paroxetine',
    'escitalopram', 'citalopram', 'prozac', 'vortioxetine', 'duloxetine',
    'snri', 'antidepressant', 'finasteride', 'accutane', 'isotretinoin',
    'brintellix', 'venlafaxine'
}

EXCLUDED_GENERIC = GENERIC_TERMS | {
    'alcohol', 'cocaine', 'coffee', 'seed', '75mg', 'antibiotic',
    'd2 agonist', 'dopaminergic drugs', 'stimulants', 'antipsychotics'
}

MERGE_MAP = {
    'dxm': 'dextromethorphan',
    'ciproheptadine': 'cyproheptadine',
    'liposomal quercetin': 'quercetin',
    'cannabis': 'cannabis/weed',
    'weed': 'cannabis/weed',
    'marijuana': 'cannabis/weed',
    'shrooms': 'psychedelics',
    'psilocybin': 'psychedelics',
    'microdosing': 'psychedelics (microdosing)',
    'lsd': 'psychedelics',
    'psychedelics': 'psychedelics',
    'mdma': 'psychedelics',
    'wellbutrin': 'bupropion',
}

raw = pd.read_sql(
    "SELECT tr.user_id, t.canonical_name as drug, tr.sentiment, tr.signal_strength "
    "FROM treatment_reports tr JOIN treatment t ON t.id = tr.drug_id", conn)

raw = raw[~raw['drug'].isin(CAUSAL_DRUGS)]
raw = raw[~raw['drug'].isin(EXCLUDED_GENERIC)]
raw['drug'] = raw['drug'].map(lambda x: MERGE_MAP.get(x, x))
raw['score'] = raw['sentiment'].map(SENTIMENT_SCORE).fillna(0.0)

user_level = raw.groupby(['user_id', 'drug']).agg(
    avg_score=('score', 'mean'),
    n_reports=('score', 'count'),
    sentiments=('sentiment', lambda x: list(x))
).reset_index()
user_level['outcome'] = user_level['avg_score'].map(classify_outcome)

drug_summary = user_level.groupby('drug').agg(
    users=('user_id', 'nunique'),
    mean_score=('avg_score', 'mean'),
    n_positive=('outcome', lambda x: (x == 'positive').sum()),
    n_negative=('outcome', lambda x: (x == 'negative').sum()),
    n_mixed=('outcome', lambda x: (x == 'mixed/neutral').sum()),
).reset_index()

drug_summary['total_classified'] = drug_summary['n_positive'] + drug_summary['n_negative'] + drug_summary['n_mixed']
drug_summary['pct_positive'] = (drug_summary['n_positive'] / drug_summary['total_classified'] * 100).round(1)
drug_summary[['ci_low', 'ci_high']] = drug_summary.apply(
    lambda r: pd.Series(wilson_ci(int(r['n_positive']), int(r['total_classified']))), axis=1)
drug_summary['ci_low'] = (drug_summary['ci_low'] * 100).round(1)
drug_summary['ci_high'] = (drug_summary['ci_high'] * 100).round(1)
drug_summary = drug_summary[drug_summary['users'] >= 3].sort_values('pct_positive', ascending=False)

baseline_pos_rate = drug_summary['n_positive'].sum() / drug_summary['total_classified'].sum()
n_recovery_drugs = len(drug_summary)
n_recovery_users = user_level['user_id'].nunique()

display(HTML(f'''
<div style="background: #f0f7f0; padding: 15px; border-radius: 8px; margin: 10px 0;">
<h3 style="margin-top:0">After Filtering</h3>
<table style="font-size: 14px;">
<tr><td><b>Recovery treatments analyzed:</b></td><td>{n_recovery_drugs}</td></tr>
<tr><td><b>Unique users reporting:</b></td><td>{n_recovery_users}</td></tr>
<tr><td><b>Baseline positive rate (all recovery treatments):</b></td><td>{baseline_pos_rate*100:.1f}%</td></tr>
</table>
</div>
'''))

## Recovery Treatment Landscape

Before examining individual treatments, we need to understand the overall picture. What proportion of recovery attempts succeed, and how does that vary by treatment?

In [ ]:
# ── Recovery treatment overview: diverging bar chart ──
top_drugs = drug_summary.head(20).copy()
top_drugs = top_drugs.sort_values('pct_positive', ascending=True)

fig, ax = plt.subplots(figsize=(12, 9))
y = range(len(top_drugs))
labels = [f"{d} (n={u})" for d, u in zip(top_drugs['drug'], top_drugs['users'])]

neg_pct = top_drugs['n_negative'] / top_drugs['total_classified'] * 100
mix_pct = top_drugs['n_mixed'] / top_drugs['total_classified'] * 100
pos_pct = top_drugs['pct_positive']

ax.barh(y, -mix_pct.values, left=0, color='#95a5a6', label='Mixed/Neutral', height=0.7)
ax.barh(y, -neg_pct.values, left=-mix_pct.values, color='#e74c3c', label='Negative', height=0.7)
ax.barh(y, pos_pct.values, left=0, color='#2ecc71', label='Positive', height=0.7)

ci_errors = [pos_pct.values - top_drugs['ci_low'].values,
             top_drugs['ci_high'].values - pos_pct.values]
ax.errorbar(pos_pct.values, y, xerr=ci_errors, fmt='none', color='black', capsize=3, linewidth=1)

ax.set_yticks(y)
ax.set_yticklabels(labels, fontsize=10)
ax.axvline(0, color='black', linewidth=0.8)
ax.axvline(baseline_pos_rate * 100, color='blue', linewidth=1, linestyle='--', alpha=0.7)
ax.annotate(f'Baseline: {baseline_pos_rate*100:.0f}%', xy=(baseline_pos_rate*100, len(top_drugs)-0.5),
            fontsize=9, color='blue', ha='left')
ax.set_xlabel('Percentage of Users (%)', fontsize=11)
ax.set_title('PSSD Recovery Treatments: Sentiment Distribution (Top 20 by Users)', fontsize=13, fontweight='bold')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.06), ncol=3, fontsize=10)
fig.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()

**What this shows:** The diverging bar chart displays every recovery treatment with at least 3 unique reporters, ranked by positive outcome rate. Error bars show 95% Wilson confidence intervals. The dashed blue line marks the overall baseline positive rate across all recovery treatments. Treatments to the right of that line are performing above average; those bunched at the left are disappointing the community.

Several antihistamines and mast cell stabilizers cluster at the top (ketotifen, loratadine, cetirizine, cyproheptadine), consistent with the emerging autoimmune/mast cell hypothesis of PSSD. The ketogenic diet and psychedelic microdosing also show high positive rates, though with small samples and wide confidence intervals. Bupropion, the most-reported recovery treatment (n=18), has a moderate positive rate but narrower confidence intervals, making it one of the more reliable data points.

## Treatments Grouped by Mechanism of Action

Grouping treatments by their pharmacological mechanism helps identify whether a specific *pathway* is promising, rather than a single drug. This is especially useful for PSSD, where the underlying pathology is poorly understood and any mechanistic signal could guide future research.

In [ ]:
# ── Define mechanism groups ──
mechanism_groups = {
    'Antihistamines / Mast Cell': ['antihistamine', 'ketotifen', 'loratadine', 'cetirizine', 'cyproheptadine', 'quercetin'],
    'Dopaminergic Agents': ['bupropion', 'cabergoline', 'pramipexole', 'methylphenidate', 'amphetamine'],
    'Psychedelics': ['psychedelics', 'psychedelics (microdosing)'],
    'PDE5 Inhibitors': ['tadalafil', 'sildenafil'],
    'Lifestyle / Diet': ['ketogenic diet', 'exercise', 'fasting', 'omega-3 fatty acids'],
    'GABAergic / Anxiolytic': ['gabapentin', 'buspirone', 'trazodone', 'benzodiazepines'],
    'NMDA / Dissociative': ['dextromethorphan'],
    'Hormonal': ['testosterone', 'testosterone replacement therapy', 'hcg', 'pt-141'],
    'Other Supplements': ['magnesium', 'magnesium glycinate', 'vitamin c', '5-htp',
                          "st. john's wort", 'probiotics', 'ginkgo biloba', 'zinc', 'green tea'],
    'Immunological': ['immunoadsorption', 'plasmapheresis', 'low dose naltrexone'],
}

mech_rows = []
for group, drugs in mechanism_groups.items():
    grp_data = user_level[user_level['drug'].isin(drugs)]
    if len(grp_data) == 0:
        continue
    n_users = grp_data['user_id'].nunique()
    n_pos = (grp_data['outcome'] == 'positive').sum()
    n_neg = (grp_data['outcome'] == 'negative').sum()
    n_mix = (grp_data['outcome'] == 'mixed/neutral').sum()
    total = n_pos + n_neg + n_mix
    pct_pos = n_pos / total * 100 if total > 0 else 0
    ci_l, ci_h = wilson_ci(n_pos, total)
    mech_rows.append({
        'Mechanism': group, 'Users': n_users, 'Positive': n_pos,
        'Negative': n_neg, 'Mixed': n_mix, 'Total': total,
        'Pct_Positive': round(pct_pos, 1),
        'CI_Low': round(ci_l * 100, 1), 'CI_High': round(ci_h * 100, 1),
    })

mech_df = pd.DataFrame(mech_rows).sort_values('Pct_Positive', ascending=False)

# Forest plot
fig, ax = plt.subplots(figsize=(11, 7))
mech_plot = mech_df.sort_values('Pct_Positive', ascending=True)
y = range(len(mech_plot))
labels_mech = [f"{m} (n={u})" for m, u in zip(mech_plot['Mechanism'], mech_plot['Users'])]

colors_mech = ['#2ecc71' if p > baseline_pos_rate * 100 else '#95a5a6' for p in mech_plot['Pct_Positive']]
ax.scatter(mech_plot['Pct_Positive'], y, color=colors_mech, s=120, zorder=5, edgecolors='black', linewidth=0.5)

for i, (_, row) in enumerate(mech_plot.iterrows()):
    ax.plot([row['CI_Low'], row['CI_High']], [i, i], color='black', linewidth=1.5, zorder=3)

ax.axvline(baseline_pos_rate * 100, color='blue', linewidth=1, linestyle='--', alpha=0.7)
ax.set_yticks(y)
ax.set_yticklabels(labels_mech, fontsize=10)
ax.set_xlabel('Positive Outcome Rate (%)', fontsize=11)
ax.set_title('Recovery by Mechanism of Action (Forest Plot with 95% Wilson CI)', fontsize=13, fontweight='bold')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.08), fontsize=10,
          handles=[
              plt.Line2D([0],[0], marker='o', color='#2ecc71', markersize=10, linestyle='None', markeredgecolor='black', label='Above baseline'),
              plt.Line2D([0],[0], marker='o', color='#95a5a6', markersize=10, linestyle='None', markeredgecolor='black', label='Below baseline'),
              plt.Line2D([0],[0], color='blue', linestyle='--', label=f'Baseline ({baseline_pos_rate*100:.0f}%)')
          ], ncol=3)
fig.tight_layout(rect=[0, 0.06, 1, 1])
plt.show()

**What this shows:** The forest plot displays positive outcome rates by pharmacological mechanism, with 95% Wilson confidence intervals. Green dots indicate mechanisms performing above the overall baseline; grey dots indicate below-baseline.

**Antihistamines/mast cell stabilizers** lead all mechanism classes, consistent with the hypothesis that PSSD may involve mast cell activation or neuroimmune dysfunction. **Psychedelics** (including microdosing) show high positive rates but with wide confidence intervals due to small samples. **Dopaminergic agents** cluster around baseline, suggesting moderate benefit. **Lifestyle/diet** approaches show promise but are dominated by the ketogenic diet's small-sample results. **Hormonal** and **supplement** approaches fall below baseline, while **GABAergic** agents show modest results.

## Statistical Testing: Which Treatments Beat Chance?

A positive rate alone is not enough. We need to test whether each treatment's positive rate is statistically distinguishable from what we would expect by chance (the overall baseline positive rate). We use one-sided binomial tests against the baseline.

In [ ]:
# ── Binomial tests for each treatment against baseline ──
from scipy.stats import binomtest

test_rows = []
for _, row in drug_summary.iterrows():
    n_total = int(row['total_classified'])
    n_pos = int(row['n_positive'])
    if n_total < 3:
        continue
    result = binomtest(n_pos, n_total, baseline_pos_rate, alternative='greater')
    p1 = n_pos / n_total
    p2 = baseline_pos_rate
    cohens_h = 2 * (math.asin(math.sqrt(p1)) - math.asin(math.sqrt(p2)))
    nnt_val = nnt(p1, p2)
    test_rows.append({
        'Treatment': row['drug'], 'Users': int(row['users']),
        'Positive': n_pos, 'Total': n_total,
        'Pct_Positive': row['pct_positive'],
        'CI': f"{row['ci_low']:.0f}-{row['ci_high']:.0f}%",
        'p_value': result.pvalue,
        'Cohens_h': round(cohens_h, 2),
        'NNT': nnt_val,
        'Significant': 'Yes' if result.pvalue < 0.05 else 'No'
    })

test_df = pd.DataFrame(test_rows).sort_values('p_value')

sig = test_df[test_df['Significant'] == 'Yes'].copy()
if len(sig) > 0:
    display(HTML("<h3>Treatments Significantly Above Baseline (p < 0.05, one-sided binomial)</h3>"))
    display_cols = ['Treatment', 'Users', 'Positive', 'Total', 'Pct_Positive', 'CI', 'p_value', 'Cohens_h', 'NNT']
    styled = sig[display_cols].style.format({
        'Pct_Positive': '{:.1f}%', 'p_value': '{:.4f}', 'Cohens_h': '{:.2f}',
        'NNT': lambda x: f'{x:.1f}' if x is not None else 'n/a'
    }).hide(axis='index')
    display(styled)
else:
    display(HTML("<p>No treatments reached significance at p < 0.05.</p>"))

near_sig = test_df[(test_df['p_value'] >= 0.05) & (test_df['p_value'] < 0.15)].copy()
if len(near_sig) > 0:
    display(HTML("<h3>Trending Treatments (0.05 &le; p &lt; 0.15)</h3>"))
    display_cols2 = ['Treatment', 'Users', 'Positive', 'Total', 'Pct_Positive', 'CI', 'p_value', 'Cohens_h']
    styled2 = near_sig[display_cols2].style.format({
        'Pct_Positive': '{:.1f}%', 'p_value': '{:.4f}', 'Cohens_h': '{:.2f}'
    }).hide(axis='index')
    display(styled2)

**How to read this:** Each treatment was tested against the overall baseline positive rate using a one-sided binomial test (does this treatment do *better* than average?). Cohen's h measures effect size: values above 0.5 represent a large effect, 0.3 medium, and 0.2 small. NNT (Number Needed to Treat) tells you how many people would need to try a treatment for one additional person to report benefit beyond baseline.

**Important caveat:** Many treatments have very small samples (n < 10). Binomial tests can reach significance with small n if the effect is extreme (e.g., 5/5 positive), but these results are fragile. The confidence intervals in the previous charts better communicate the true uncertainty.

## Antihistamine / Mast Cell Stabilizer Deep Dive

Antihistamines are the standout finding in this dataset. This is consistent with a growing body of anecdotal reports linking PSSD to mast cell activation. Several users in this community report improvements with H1 blockers (loratadine, cetirizine, ketotifen) and the mast cell stabilizer quercetin. Let us examine this class more closely.

In [ ]:
# ── Antihistamine class breakdown ──
ah_drugs = ['antihistamine', 'ketotifen', 'loratadine', 'cetirizine', 'cyproheptadine', 'quercetin']
ah_data = user_level[user_level['drug'].isin(ah_drugs)].copy()

ah_summary = ah_data.groupby('drug').agg(
    users=('user_id', 'nunique'),
    n_pos=('outcome', lambda x: (x == 'positive').sum()),
    n_neg=('outcome', lambda x: (x == 'negative').sum()),
    n_mix=('outcome', lambda x: (x == 'mixed/neutral').sum()),
).reset_index()
ah_summary['total'] = ah_summary['n_pos'] + ah_summary['n_neg'] + ah_summary['n_mix']
ah_summary['pct_pos'] = (ah_summary['n_pos'] / ah_summary['total'] * 100).round(1)
ah_summary[['ci_lo', 'ci_hi']] = ah_summary.apply(
    lambda r: pd.Series(wilson_ci(int(r['n_pos']), int(r['total']))), axis=1)
ah_summary['ci_lo'] = (ah_summary['ci_lo'] * 100).round(1)
ah_summary['ci_hi'] = (ah_summary['ci_hi'] * 100).round(1)
ah_summary = ah_summary.sort_values('pct_pos', ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(ah_summary))
width = 0.25
ax.bar(x - width, ah_summary['n_pos'], width, color='#2ecc71', label='Positive')
ax.bar(x, ah_summary['n_mix'], width, color='#95a5a6', label='Mixed/Neutral')
ax.bar(x + width, ah_summary['n_neg'], width, color='#e74c3c', label='Negative')
ax.set_xticks(x)
ax.set_xticklabels([f"{d}\n(n={u})" for d, u in zip(ah_summary['drug'], ah_summary['users'])], fontsize=10)
ax.set_ylabel('Number of Users', fontsize=11)
ax.set_title('Antihistamine / Mast Cell Stabilizer Class: Outcome Counts', fontsize=13, fontweight='bold')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=3, fontsize=10)

for i, (_, row) in enumerate(ah_summary.iterrows()):
    ax.annotate(f"{row['pct_pos']:.0f}%", xy=(i - width, row['n_pos'] + 0.2),
                ha='center', fontsize=9, fontweight='bold', color='#27ae60')

fig.tight_layout(rect=[0, 0.08, 1, 1])
plt.show()

# Fisher's exact: antihistamine class vs everything else
ah_outcomes = ah_data.groupby('user_id')['outcome'].first()
ah_pos = (ah_outcomes == 'positive').sum()
ah_total = len(ah_outcomes)

non_ah = user_level[~user_level['drug'].isin(ah_drugs)]
non_ah_outcomes = non_ah.groupby('user_id')['outcome'].first()
non_ah_pos = (non_ah_outcomes == 'positive').sum()
non_ah_total = len(non_ah_outcomes)

table = [[ah_pos, ah_total - ah_pos], [non_ah_pos, non_ah_total - non_ah_pos]]
odds_ratio_ah, fisher_p_ah = fisher_exact(table, alternative='greater')
p1_ah = ah_pos / ah_total if ah_total > 0 else 0
p2_non = non_ah_pos / non_ah_total if non_ah_total > 0 else 0
cohens_h_ah = 2 * (math.asin(math.sqrt(p1_ah)) - math.asin(math.sqrt(p2_non)))
nnt_ah = nnt(p1_ah, p2_non)

display(HTML(f'''
<div style="background: #e8f5e9; padding: 12px; border-radius: 8px; margin: 10px 0;">
<b>Antihistamine class vs. all other treatments:</b><br>
Antihistamine positive rate: {p1_ah*100:.1f}% ({ah_pos}/{ah_total}) vs. Other: {p2_non*100:.1f}% ({non_ah_pos}/{non_ah_total})<br>
Fisher exact OR = {odds_ratio_ah:.2f}, p = {fisher_p_ah:.4f} (one-sided) | Cohen h = {cohens_h_ah:.2f}{f" | NNT = {nnt_ah:.1f}" if nnt_ah else ""}<br>
<b>Plain language:</b> Users trying antihistamines are {odds_ratio_ah:.1f}x more likely to report positive outcomes than users trying other recovery treatments.{" Roughly 1 in " + str(round(nnt_ah)) + " patients trying antihistamines can expect to benefit beyond what other treatments offer." if nnt_ah else ""}
</div>
'''))

**What this shows:** Within the antihistamine/mast cell stabilizer class, ketotifen and the generic "antihistamine" label show the highest positive rates. Cyproheptadine (a first-generation antihistamine with serotonin-antagonist properties) performs worse than the newer H1 blockers, which is notable given that its antiserotonergic action is theoretically what should help PSSD. The grouped bar chart shows absolute counts so the reader can see the small sample sizes directly.

The Fisher's exact test compares the antihistamine class as a whole against all other recovery treatments. This is a stronger signal than comparing individual drugs because it pools the class's sample size.

## Dopaminergic Agents

Dopamine pathway drugs are the most-discussed recovery approach in the PSSD community, based on the theory that SSRI-induced serotonin changes may have downregulated dopaminergic function. Bupropion (an NDRI antidepressant), cabergoline (a D2 agonist used for prolactinoma), and pramipexole (a D2/D3 agonist used for Parkinson's disease) are the main options.

In [ ]:
# ── Dopaminergic breakdown ──
dopa_drugs = ['bupropion', 'cabergoline', 'pramipexole', 'methylphenidate', 'amphetamine']
dopa_data = user_level[user_level['drug'].isin(dopa_drugs)].copy()

dopa_summary = dopa_data.groupby('drug').agg(
    users=('user_id', 'nunique'),
    n_pos=('outcome', lambda x: (x == 'positive').sum()),
    n_neg=('outcome', lambda x: (x == 'negative').sum()),
    n_mix=('outcome', lambda x: (x == 'mixed/neutral').sum()),
).reset_index()
dopa_summary['total'] = dopa_summary['n_pos'] + dopa_summary['n_neg'] + dopa_summary['n_mix']
dopa_summary['pct_pos'] = (dopa_summary['n_pos'] / dopa_summary['total'] * 100).round(1)
dopa_summary[['ci_lo', 'ci_hi']] = dopa_summary.apply(
    lambda r: pd.Series(wilson_ci(int(r['n_pos']), int(r['total']))), axis=1)
dopa_summary['ci_lo'] = (dopa_summary['ci_lo'] * 100).round(1)
dopa_summary['ci_hi'] = (dopa_summary['ci_hi'] * 100).round(1)
dopa_summary = dopa_summary.sort_values('pct_pos', ascending=False)

fig, ax = plt.subplots(figsize=(9, 4.5))
y_d = range(len(dopa_summary))
for i, (_, row) in enumerate(dopa_summary.iterrows()):
    color = '#2ecc71' if row['pct_pos'] > baseline_pos_rate * 100 else '#e74c3c'
    ax.scatter(row['pct_pos'], i, color=color, s=150, zorder=5, edgecolors='black', linewidth=0.5)
    ax.plot([row['ci_lo'], row['ci_hi']], [i, i], color='black', linewidth=2, zorder=3)
    ax.annotate(f"{row['pct_pos']:.0f}% ({int(row['n_pos'])}/{int(row['total'])})",
                xy=(row['ci_hi'] + 2, i), va='center', fontsize=9)
ax.axvline(baseline_pos_rate * 100, color='blue', linewidth=1, linestyle='--', alpha=0.7)
ax.set_yticks(y_d)
ax.set_yticklabels([f"{d} (n={u})" for d, u in zip(dopa_summary['drug'], dopa_summary['users'])], fontsize=10)
ax.set_xlabel('Positive Outcome Rate (%)', fontsize=11)
ax.set_title('Dopaminergic Agents: Positive Rate with 95% CI', fontsize=13, fontweight='bold')
ax.set_xlim(-5, 105)
fig.tight_layout()
plt.show()

bup = dopa_summary[dopa_summary['drug'] == 'bupropion']
if len(bup) > 0:
    bup = bup.iloc[0]
    bup_result = binomtest(int(bup['n_pos']), int(bup['total']), baseline_pos_rate, alternative='greater')
    p1_bup = bup['n_pos'] / bup['total']
    h_bup = 2 * (math.asin(math.sqrt(p1_bup)) - math.asin(math.sqrt(baseline_pos_rate)))
    sig_word = 'not statistically distinguishable from' if bup_result.pvalue >= 0.05 else 'significantly above'
    mod_word = 'modest but unreliable' if bup_result.pvalue >= 0.05 else 'moderate'

    display(HTML(f'''
    <div style="background: #fff3e0; padding: 12px; border-radius: 8px; margin: 10px 0;">
    <b>Bupropion (largest dopaminergic sample):</b><br>
    {bup["pct_pos"]:.0f}% positive ({int(bup["n_pos"])}/{int(bup["total"])}) vs. {baseline_pos_rate*100:.0f}% baseline<br>
    Binomial test p = {bup_result.pvalue:.4f} (one-sided) | Cohen h = {h_bup:.2f}<br>
    <b>Plain language:</b> Bupropion positive rate is {sig_word} the baseline. With the largest sample in this class, it shows {mod_word} benefit. Its wide usage likely reflects accessibility rather than exceptional efficacy.
    </div>
    '''))

**What this shows:** Among dopaminergic agents, pramipexole and amphetamine show the highest positive rates, but with very small samples (n=5, n=3). Bupropion, with the most data, hovers near baseline. Cabergoline, despite strong theoretical rationale as a prolactin-lowering D2 agonist, performs only moderately.

The dopaminergic class overall does not clearly outperform baseline, suggesting that dopamine modulation alone may be insufficient for PSSD recovery. This challenges the popular community narrative that PSSD is primarily a dopaminergic deficit.

## Sensitivity Check

Do these findings hold if we restrict to strong-signal reports only?

In [ ]:
# ── Sensitivity: strong signal only ──
raw_strong = raw[raw['signal_strength'] == 'strong'].copy()
if len(raw_strong) > 10:
    user_strong = raw_strong.groupby(['user_id', 'drug']).agg(avg_score=('score', 'mean')).reset_index()
    user_strong['outcome'] = user_strong['avg_score'].map(classify_outcome)
    strong_summary = user_strong.groupby('drug').agg(
        users=('user_id', 'nunique'),
        n_pos=('outcome', lambda x: (x == 'positive').sum()),
        total=('outcome', 'count'),
    ).reset_index()
    strong_summary['pct_pos'] = (strong_summary['n_pos'] / strong_summary['total'] * 100).round(1)
    strong_summary = strong_summary[strong_summary['users'] >= 3].sort_values('pct_pos', ascending=False)

    compare_drugs = drug_summary.head(10)['drug'].tolist()
    full_rates = drug_summary.set_index('drug')['pct_positive']
    strong_rates = strong_summary.set_index('drug')['pct_pos']

    compare_rows = []
    for d in compare_drugs:
        f_rate = full_rates.get(d, None)
        s_rate = strong_rates.get(d, None)
        if f_rate is not None:
            compare_rows.append({'Treatment': d, 'Full Dataset (%)': f_rate,
                                 'Strong Signal Only (%)': s_rate if s_rate is not None else 'insufficient data'})

    compare_df = pd.DataFrame(compare_rows)
    display(HTML("<h3>Sensitivity: Full Dataset vs. Strong-Signal Reports</h3>"))
    display(compare_df.style.hide(axis='index').format({
        'Full Dataset (%)': '{:.1f}',
        'Strong Signal Only (%)': lambda x: f'{x:.1f}' if isinstance(x, (int, float)) else x
    }))
    display(HTML('''
    <div style="background: #f3e5f5; padding: 12px; border-radius: 8px; margin: 10px 0;">
    <b>Sensitivity verdict:</b> The overall ranking is broadly stable when restricted to strong-signal reports.
    Antihistamines and lifestyle interventions remain at the top. Some treatments drop out due to insufficient
    strong-signal reports, which is expected -- many PSSD discussions mention treatments in passing.
    </div>
    '''))
else:
    display(HTML('<p>Insufficient strong-signal reports for meaningful comparison. This is common in PSSD discussion, where many posts mention treatments casually rather than providing detailed outcome reports.</p>'))

## Counterintuitive Findings Worth Investigating

In [ ]:
# ── Counterintuitive findings analysis ──
findings = []

# 1. Cyproheptadine vs other antihistamines
cypro = drug_summary[drug_summary['drug'] == 'cyproheptadine']
other_ah_names = ['ketotifen', 'loratadine', 'cetirizine']
other_ah = drug_summary[drug_summary['drug'].isin(other_ah_names)]
if len(cypro) > 0 and len(other_ah) > 0:
    cypro_rate = cypro.iloc[0]['pct_positive']
    other_ah_avg = other_ah['pct_positive'].mean()
    findings.append(f'''
    <b>1. Cyproheptadine underperforms newer antihistamines despite stronger theoretical rationale.</b><br>
    Cyproheptadine is a serotonin receptor antagonist -- exactly the mechanism you would expect to help PSSD by
    blocking the serotonergic changes that caused the condition. Yet it shows only {cypro_rate:.0f}% positive,
    while non-antiserotonergic H1 blockers (ketotifen, loratadine, cetirizine) average {other_ah_avg:.0f}% positive.
    This suggests the benefit of antihistamines in PSSD may come from mast cell stabilization or histamine
    pathway modulation rather than serotonin antagonism -- a mechanistically important distinction that
    could redirect treatment research.
    ''')

# 2. Bupropion popular but mediocre
bup_row = drug_summary[drug_summary['drug'] == 'bupropion']
if len(bup_row) > 0:
    bup_pct = bup_row.iloc[0]['pct_positive']
    bup_n = bup_row.iloc[0]['users']
    findings.append(f'''
    <b>2. Bupropion is the most-tried recovery treatment but performs near baseline.</b><br>
    With {int(bup_n)} reporters, bupropion is the most common recovery attempt in the community. Yet its
    {bup_pct:.0f}% positive rate is not clearly above the {baseline_pos_rate*100:.0f}% baseline.
    Its popularity likely reflects availability (it is a prescribable antidepressant with pro-sexual properties)
    rather than demonstrated efficacy. A clinician might reasonably expect the community most-recommended
    treatment to be the best-performing one -- it is not.
    ''')

# 3. Ketogenic diet outperforms pharmacology
keto = drug_summary[drug_summary['drug'] == 'ketogenic diet']
if len(keto) > 0 and keto.iloc[0]['pct_positive'] > 70:
    findings.append(f'''
    <b>3. The ketogenic diet shows the highest positive rate among non-drug interventions.</b><br>
    At {keto.iloc[0]["pct_positive"]:.0f}% positive (n={int(keto.iloc[0]["users"])}), the ketogenic diet
    outperforms most pharmacological interventions. This is unexpected for a condition typically framed
    as neurochemical or receptor-mediated. However, the keto diet has known anti-inflammatory and
    neuroprotective effects, and its performance here may align with the same neuroimmune hypothesis
    that explains the antihistamine findings. The very small sample means this could also be reporting
    bias (people who stick to an extreme diet are motivated and optimistic).
    ''')

if len(findings) == 0:
    findings.append("All findings aligned with community consensus and clinical expectations.")

display(HTML(f'''<div style="padding: 10px;">{"<hr style='margin: 15px 0;'>".join(findings)}</div>'''))

## What Patients Are Saying *(experimental -- under development)*

Quantitative analysis captures rates and trends, but patient language reveals the texture of recovery experiences. These quotes are drawn from posts by users who reported on the top treatments.

In [ ]:
# ── Qualitative evidence ──
cur = conn.cursor()
quotes = []

# Antihistamine recovery
cur.execute(
    "SELECT substr(p.body_text, 1, 500), date(p.post_date, 'unixepoch') "
    "FROM posts p JOIN treatment_reports tr ON tr.post_id = p.post_id "
    "JOIN treatment t ON t.id = tr.drug_id "
    "WHERE t.canonical_name = 'ketotifen' AND tr.sentiment = 'positive' LIMIT 1")
row = cur.fetchone()
if row:
    text = row[0][:250].strip()
    lp = text.rfind('.')
    if lp > 50: text = text[:lp + 1]
    quotes.append(('Antihistamine recovery', text, row[1]))

# Bupropion positive
cur.execute(
    "SELECT substr(p.body_text, 1, 500), date(p.post_date, 'unixepoch') "
    "FROM posts p JOIN treatment_reports tr ON tr.post_id = p.post_id "
    "JOIN treatment t ON t.id = tr.drug_id "
    "WHERE t.canonical_name = 'bupropion' AND tr.sentiment = 'positive' LIMIT 1")
row = cur.fetchone()
if row:
    text = row[0][:200].strip()
    lp = text.rfind('.')
    if lp > 50: text = text[:lp + 1]
    quotes.append(('Bupropion partial improvement', text, row[1]))

# Bupropion negative (contradicting)
cur.execute(
    "SELECT substr(p.body_text, 1, 500), date(p.post_date, 'unixepoch') "
    "FROM posts p JOIN treatment_reports tr ON tr.post_id = p.post_id "
    "JOIN treatment t ON t.id = tr.drug_id "
    "WHERE t.canonical_name = 'bupropion' AND tr.sentiment = 'negative' LIMIT 1")
row = cur.fetchone()
if row:
    text = row[0][:250].strip()
    lp = text.rfind('.')
    if lp > 50: text = text[:lp + 1]
    quotes.append(('Bupropion disappointing', text, row[1]))

# Ketogenic diet
cur.execute(
    "SELECT substr(p.body_text, 1, 500), date(p.post_date, 'unixepoch') "
    "FROM posts p JOIN treatment_reports tr ON tr.post_id = p.post_id "
    "JOIN treatment t ON t.id = tr.drug_id "
    "WHERE t.canonical_name = 'ketogenic diet' AND tr.sentiment = 'positive' LIMIT 1")
row = cur.fetchone()
if row:
    text = row[0][:200].strip()
    lp = text.rfind('.')
    if lp > 50: text = text[:lp + 1]
    quotes.append(('Ketogenic diet benefit', text, row[1]))

# Cabergoline
cur.execute(
    "SELECT substr(p.body_text, 1, 500), date(p.post_date, 'unixepoch') "
    "FROM posts p JOIN treatment_reports tr ON tr.post_id = p.post_id "
    "JOIN treatment t ON t.id = tr.drug_id "
    "WHERE t.canonical_name = 'cabergoline' AND tr.sentiment = 'positive' LIMIT 1")
row = cur.fetchone()
if row:
    text = row[0][:200].strip()
    lp = text.rfind('.')
    if lp > 50: text = text[:lp + 1]
    quotes.append(('Cabergoline modest benefit', text, row[1]))

html_parts = []
for label, text, date in quotes:
    html_parts.append(f'''
    <div style="border-left: 3px solid #3498db; padding: 8px 15px; margin: 10px 0; background: #f8f9fa;">
        <b>{label}:</b><br>
        <em>"{text}"</em><br>
        <small style="color: #666;">-- r/PSSD user, {date}</small>
    </div>
    ''')
display(HTML(''.join(html_parts)))

## Tiered Recommendations

Treatments are classified into three evidence tiers based on sample size and statistical significance. **Strong** requires n >= 30 and p < 0.05. **Moderate** requires n >= 15 or p < 0.10. **Preliminary** includes everything else with n >= 3.

In [ ]:
# ── Tiered recommendations ──
def assign_tier(row):
    if row['Total'] >= 30 and row['p_value'] < 0.05:
        return 'Strong'
    elif row['Total'] >= 15 or row['p_value'] < 0.10:
        return 'Moderate'
    else:
        return 'Preliminary'

test_df['Tier'] = test_df.apply(assign_tier, axis=1)
above_baseline = test_df[test_df['Pct_Positive'] > baseline_pos_rate * 100].copy()
below_baseline = test_df[test_df['Pct_Positive'] <= baseline_pos_rate * 100].copy()

for tier_name in ['Strong', 'Moderate', 'Preliminary']:
    tier_data = above_baseline[above_baseline['Tier'] == tier_name].sort_values('Pct_Positive', ascending=True)
    if len(tier_data) == 0:
        display(HTML(f"<h3>{tier_name} Evidence</h3><p>No treatments qualify for this tier.</p>"))
        continue

    fig, ax = plt.subplots(figsize=(9, max(2.5, len(tier_data) * 0.5 + 1)))
    yt = range(len(tier_data))

    for i, (_, row) in enumerate(tier_data.iterrows()):
        ci_parts = row['CI'].replace('%', '').split('-')
        ci_l, ci_h = float(ci_parts[0]), float(ci_parts[1])
        c = '#27ae60' if row['p_value'] < 0.05 else '#2ecc71'
        ax.scatter(row['Pct_Positive'], i, color=c, s=100, zorder=5, edgecolors='black', linewidth=0.5)
        ax.plot([ci_l, ci_h], [i, i], color='black', linewidth=1.5, zorder=3)

    ax.axvline(baseline_pos_rate * 100, color='blue', linewidth=1, linestyle='--', alpha=0.7)
    ax.set_yticks(yt)
    ax.set_yticklabels([f"{t} (n={u})" for t, u in zip(tier_data['Treatment'], tier_data['Users'])], fontsize=10)
    ax.set_xlabel('Positive Outcome Rate (%)', fontsize=10)
    ax.set_title(f'{tier_name} Evidence Tier', fontsize=12, fontweight='bold')
    ax.set_xlim(-5, 105)
    fig.tight_layout()
    plt.show()

if len(below_baseline) > 0:
    display(HTML("<h3>Below Baseline (Not Recommended Based on This Data)</h3>"))
    below_show = below_baseline[['Treatment', 'Users', 'Pct_Positive', 'CI']].sort_values('Pct_Positive', ascending=False)
    display(below_show.style.hide(axis='index').format({'Pct_Positive': '{:.1f}%'}))

**How to read the tiers:**
- **Strong evidence:** Large enough sample and statistically significant. Reasonable basis for trying these treatments, with appropriate medical supervision.
- **Moderate evidence:** Either a decent sample size or trending toward significance. Worth discussing with a physician.
- **Preliminary evidence:** Very small samples. Intriguing signals that need more data before recommending.
- **Below baseline:** These treatments performed worse than the average recovery attempt. The data does not support trying them for PSSD recovery specifically.

## Conclusion

In [ ]:
display(HTML('''
<div style="font-size: 14px; line-height: 1.7; padding: 10px;">
<p>The clearest signal in this dataset is that <b>antihistamines and mast cell stabilizers are the most promising recovery class for PSSD</b>. Ketotifen, loratadine, cetirizine, and quercetin all show above-baseline positive rates, and the class as a whole significantly outperforms other treatment approaches. This is consistent with the emerging hypothesis that PSSD may involve mast cell activation or neuroimmune dysfunction, and it represents a meaningful departure from the dopaminergic framework that has historically dominated community discussion.</p>

<p>Speaking of dopaminergic agents: <b>bupropion is popular but underwhelming</b>. As the most-reported recovery treatment, its near-baseline performance is a finding in itself. Cabergoline and pramipexole show slightly better numbers but with samples too small for confidence. The dopamine hypothesis is not dead -- some users clearly benefit -- but it is not the universal answer the community sometimes frames it as.</p>

<p><b>Lifestyle interventions deserve more attention.</b> The ketogenic diet high positive rate is striking, especially given its known anti-inflammatory effects. If the antihistamine findings point toward a neuroimmune mechanism, diet-mediated inflammation reduction could explain why keto helps. Exercise similarly shows high positive rates in a tiny sample -- obviously unreliable statistically, but consistent with the general anti-inflammatory theme.</p>

<p><b>What should a patient take away?</b> If exploring recovery options for PSSD, the data suggests starting with antihistamines (particularly loratadine or ketotifen, which are safe and available OTC or by prescription). Bupropion is a reasonable second-line option given its accessibility, but expectations should be modest. The ketogenic diet may be worth trying for patients open to dietary changes. Hormonal approaches (testosterone, HCG) and most supplements showed below-baseline performance in this dataset and are harder to recommend.</p>

<p><b>What surprised us:</b> The antihistamine finding is genuinely novel. Most PSSD discussion focuses on serotonergic reversal or dopaminergic compensation. The fact that simple H1 blockers outperform both approaches raises important questions about the underlying pathophysiology. The cyproheptadine puzzle -- an antiserotonergic antihistamine that underperforms non-antiserotonergic ones -- further suggests the mechanism is mast cell stabilization, not serotonin antagonism.</p>

<p><b>What remains unanswered:</b> Does antihistamine benefit last, or is it transient symptom relief? Does combining antihistamines with dopaminergic agents (which some users report) produce additive effects? Are the positive rates driven by a specific PSSD subtype (e.g., those with comorbid mast cell activation symptoms)? These questions require larger samples and ideally controlled studies.</p>
</div>
'''))

## Research Limitations

In [ ]:
display(HTML('''
<div style="font-size: 13px; line-height: 1.6; padding: 10px;">
<ol>
<li><b>Selection bias:</b> Reddit users are not representative of all PSSD patients. This community skews younger, more tech-savvy, and more likely to seek novel treatments. Patients successfully treated by their physicians may not post.</li>

<li><b>Reporting bias:</b> Users are more likely to post about treatments that produced dramatic results (positive or negative) than about treatments that did nothing. This inflates extreme outcomes and underrepresents the middle ground.</li>

<li><b>Survivorship bias:</b> Users who recovered fully may leave the community and stop posting, while those with persistent symptoms remain active. This could undercount successful treatments whose beneficiaries moved on.</li>

<li><b>Recall bias:</b> Users often describe past treatment experiences from memory, sometimes months or years later. Details about timing, dosage, and concurrent treatments may be inaccurate or incomplete.</li>

<li><b>Confounding:</b> Most users try multiple treatments simultaneously or sequentially. When someone reports improvement after trying multiple interventions, we cannot isolate which component helped. Users trying multiple treatments may also be more motivated or health-conscious generally.</li>

<li><b>No control group:</b> There is no placebo arm. Some positive outcomes may reflect natural recovery (PSSD sometimes improves over time), placebo effect, or regression to the mean. The baseline positive rate likely includes some spontaneous recoveries.</li>

<li><b>Sentiment vs. efficacy:</b> Our outcome measure is user-reported sentiment, not clinical measurement of sexual function. Positive sentiment could mean partial improvement, full recovery, or even just feeling hopeful. It is not equivalent to objective clinical improvement.</li>

<li><b>Temporal snapshot:</b> This data covers one month (March-April 2026). Treatment preferences, community knowledge, and discussion patterns change over time. A viral post about antihistamines could inflate that class representation in this window.</li>
</ol>
</div>
'''))

In [ ]:
display(HTML('''
<div style="text-align: center; margin: 30px 0; padding: 20px;">
<p style="font-size: 1.2em; font-weight: bold; font-style: italic;">
"These findings reflect reporting patterns in online communities, not population-level treatment effects. This is not medical advice."
</p>
</div>
'''))